In [1]:
import pandas as pd
from scipy import stats

# 1. Load the data
# Mapping: Which model corresponds to Option A, B, C, D for each question
mapping_df = pd.read_csv('Summary Ranking Mapping - Best & Second Best - Question Mapping.csv')
# Responses: User choices for BEST and SECOND BEST
survey_df = pd.read_csv('Summary_Evaluation_Study_-_Best___Second_Best.csv')

# 2. Extract unique models and set question count
models_list = ['Fact', 'Hierarchical', 'NLI', 'Original']
num_questions = 20

# 3. Map and Score responses
processed_data = []

for q_idx in range(num_questions):
    q_num = q_idx + 1
    # Get model mapping for this question
    m_row = mapping_df[mapping_df['Question Number'] == q_num].iloc[0]
    model_map = {'A': m_row['Option A Model'], 
                 'B': m_row['Option B Model'], 
                 'C': m_row['Option C Model'], 
                 'D': m_row['Option D Model']}
    
    # Identify survey columns for this question index
    suffix = f".{q_idx}" if q_idx > 0 else ""
    best_col = f"Which option is the BEST summary for this document?{suffix}"
    second_best_col = f"Which option is the SECOND BEST summary for this document?{suffix}"
    
    for _, response in survey_df.iterrows():
        best_choice = str(response[best_col]).strip()
        second_choice = str(response[second_best_col]).strip()
        
        # Check every model for this question and assign points
        for opt_letter, model_name in model_map.items():
            score = 0
            is_best = False
            is_second = False
            
            if opt_letter == best_choice:
                score = 2
                is_best = True
            elif opt_letter == second_choice:
                score = 1
                is_second = True
                
            processed_data.append({
                'User': response['Name'],
                'Question': q_num,
                'Model': model_name,
                'Score': score,
                'Rank': "Best" if is_best else ("Second Best" if is_second else "None")
            })

# Create final DataFrame
analysis_df = pd.DataFrame(processed_data)

# 4. Statistical Analysis
print("--- Form 3: Best and Second Best Analysis ---")

# Summary of Counts
best_counts = analysis_df[analysis_df['Rank'] == "Best"].groupby('Model').size()
second_counts = analysis_df[analysis_df['Rank'] == "Second Best"].groupby('Model').size()

print("\nFrequency Table:")
summary_table = pd.DataFrame({'Times Best': best_counts, 'Times Second Best': second_counts})
print(summary_table)

# Mean Weighted Score
print("\nMean Weighted Score (Best=2, Second=1):")
print(analysis_df.groupby('Model')['Score'].mean())

# Kruskal-Wallis H-test
data_arrays = [analysis_df[analysis_df['Model'] == m]['Score'] for m in models_list]
h_stat, p_val = stats.kruskal(*data_arrays)

print(f"\nKruskal-Wallis H-Statistic: {h_stat:.4f}")
print(f"P-Value: {p_val:.4e}")

if p_val < 0.05:
    print("Result: Significant difference! The models are ranked differently by users.")
else:
    print("Result: No significant difference in rankings.")

# Save the processed data
analysis_df.to_csv('Processed_Best_SecondBest_Scores.csv', index=False)

--- Form 3: Best and Second Best Analysis ---

Frequency Table:
              Times Best  Times Second Best
Model                                      
Fact                  12                 11
Hierarchical          16                 13
NLI                   12                 17
Original              20                 19

Mean Weighted Score (Best=2, Second=1):
Model
Fact            0.583333
Hierarchical    0.750000
NLI             0.683333
Original        0.983333
Name: Score, dtype: float64

Kruskal-Wallis H-Statistic: 8.0404
P-Value: 4.5183e-02
Result: Significant difference! The models are ranked differently by users.
